# Ballé 2018 scale-hyperprior analysis
Read saved artifacts from the method-local run and evaluation directories. Model/training logic lives in the Python package.

Protocol: full-image RGB metrics on clamped decoded float outputs; no eight-pixel crop. The local pilot trained from scratch on DIV2K for approximately 90 minutes. This is one operating point, not a converged paper reproduction.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'balle2018').is_dir() and (p / 'variable_rate').is_dir())
sys.path.insert(0, str(ROOT))
from balle2018.runtime import configure_cache
configure_cache('baseline')
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from balle2018.analysis import plot_rd
RUN = ROOT / 'balle2018/runs/div2k-pilot'
EVALUATION = ROOT / 'balle2018/results/div2k-pilot'
summary_path = EVALUATION / 'summary.csv'
summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
print('Loaded', len(summary), 'operating points. Update RUN/EVALUATION for another experiment.')

## Training and validation
Validation measures unclamped synthesis distortion, as used in training. Rate includes padding symbols; distortion uses original pixels. The pilot's validation images are fixed 512x512 center crops of 40 held-out originals.

In [ ]:
if (RUN / 'train.jsonl').exists():
    train = pd.read_json(RUN / 'train.jsonl', lines=True)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    train.plot(x='step', y='loss', ax=axes[0], title='Sampled training RD loss')
    train.plot(x='step', y=['y_bpp', 'z_bpp'], ax=axes[1], title='Estimated rates')
    plt.tight_layout()
    if (RUN / 'validation.jsonl').exists():
        validation = pd.read_json(RUN / 'validation.jsonl', lines=True)
        validation.plot(x='step', y='loss', title='Held-out validation RD loss')
    plt.show()
else:
    print('No local training logs. See balle2018/README.md.')

## Rate, distortion, and model cost
File bpp includes both entropy streams and the header. Estimated bpp should be compared with payload bpp. Parameter bytes exclude optimizer state; checkpoint size includes it.

In [ ]:
if not summary.empty:
    display(summary[['method', 'lambda', 'file_bpp', 'payload_bpp', 'estimated_bpp', 'psnr', 'ms_ssim', 'parameters', 'encode_seconds', 'decode_seconds']])
    plot_rd(summary)
    plt.show()
    metadata = json.loads((EVALUATION / 'summary.json').read_text())['metadata']
    print('Evaluation protocol:', metadata)
else:
    print('No evaluation artifacts. Run the documented evaluator first.')

## Reconstruction inspection
PNG previews are separately rounded to eight bits. Reported metrics use decoded float tensors. Set BOX to a shared (left, top, right, bottom) crop to inspect detail.

In [ ]:
ORIGINAL = ROOT / 'balle2018/data/kodak/kodim23.png'
RECONSTRUCTION = next(EVALUATION.glob('*/q0-d1/*-kodim23.png'), None)
BOX = None
if ORIGINAL.exists() and RECONSTRUCTION is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, path, title in zip(axes, [ORIGINAL, RECONSTRUCTION], ['Original', 'Decoded']):
        with Image.open(path) as image:
            ax.imshow(image.crop(BOX) if BOX else image)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Original/reconstruction files are local artifacts; prepare data and evaluate first.')

## Reproduction notes
Record dataset release and split hashes, N/M widths, lambda convention, training updates, batch/crop size, seed, device, convergence evidence, and library versions. The objective is $R_y+R_z+\lambda 255^2\,\mathrm{MSE}$. Keep this full-image RGB protocol explicit when comparing with the team's cropped-luma experiments.